# Uncertain Futures: build a three-stock portfolio
**Teams of 2–3 · approximately 25 minutes**

You have **10,000 fictional tokens**. Allocate among A, B, C, and cash; hold for **20 trading days**. No leverage, shorting, or rebalancing. Fractional shares are allowed. Cash earns zero interest; there are no trading costs.

Your job is to make a defensible decision under uncertainty. The final profit leaderboard measures one realized outcome, not proof of forecasting skill.

Only historical data are supplied. Do not try to retrieve the hidden future or the simulator. Use your agent to investigate assumptions and validation.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'workshop').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})
DATA = ROOT / 'data' / 'public'


In [ ]:
from workshop.stocks import forecast, prior_paths, validation, portfolio_scenarios, mixture_forecast, ASSETS
history = pd.read_csv(DATA / 'stock_history.csv')
history.plot(x='day', y=ASSETS)
plt.ylabel('Price'); plt.title('The information available at the investment date'); plt.show()
display(history.tail())


## 1. A prior over functions
We model **log price** as a mean function plus a GP, with independent Gaussian observation noise:
$$\log P_t=m(t)+f(t)+\epsilon_t,\quad f\sim GP(0,k),\quad\epsilon_t\sim N(0,\sigma^2).$$
The kernel describes how function values relate across time. Its length scale and amplitude specify prior beliefs about smoothness and variation. Draw functions before fitting.

Predict first: which kernel will permit abrupt-looking changes?


In [ ]:
days = np.arange(100)
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, kernel in zip(axes, ['rbf', 'matern', 'rough']):
    ax.plot(days, prior_paths(days, kernel=kernel)); ax.set_title(kernel); ax.set_xlabel('Day')
axes[0].set_ylabel('Log-price residual'); plt.tight_layout(); plt.show()


## 2. Explore one forecast
The core path fixes kernel hyperparameters: we condition exactly on these chosen assumptions. This is **not** full Bayesian integration over kernel parameters. The optional mixture below puts an explicit discrete prior on length scale.

The mean is either flat (anchored at the last observed log price) or a fitted linear trend. Both are data-estimated choices; uncertainty in the fitted mean coefficients is omitted.

For stationary kernels, forecasts far from observations return toward the assumed mean; variance approaches the prior variance plus observation noise. It does not necessarily keep growing with time.

The plotted line is a **median price**, not an expected price. The band is a pointwise 90% predictive interval including observation noise; it is not a simultaneous band for an entire path.


In [ ]:
ASSET = 'A'
KERNEL, LENGTH, AMPLITUDE, NOISE, MEAN = 'matern', 25., .15, .015, 'flat'  # EDIT
settings = dict(kernel=KERNEL, length=LENGTH, amplitude=AMPLITUDE, noise=NOISE, mean=MEAN)
f = forecast(history, ASSET, **settings)
median, low, high = f.price_interval()
plt.plot(history.day, history[ASSET], color='black', label='History')
plt.plot(f.day, median, label='Predictive median')
plt.fill_between(f.day, low, high, alpha=.25, label='90% predictive interval')
paths = np.exp(np.random.default_rng(4).multivariate_normal(f.mean, f.covariance, size=5))
plt.plot(f.day, paths.T, alpha=.35, linewidth=1)
plt.xlabel('Day'); plt.ylabel('Price'); plt.legend(); plt.show()


## 3. Test on the past before committing
Use expanding chronological training windows, with 20-day endpoint forecasts. No random train/test split. Every fit, mean estimate, and baseline variance uses the training window only.

A random-walk baseline predicts unchanged log price, with log-return variance increasing linearly with horizon. This baseline also makes assumptions.

Compare error, coverage, interval width, and log predictive density (larger log score is better). Four windows give only a noisy diagnostic; **they do not establish calibration**. Trying many configurations on these same windows can overfit validation.


In [ ]:
checks = validation(history, ASSET, **settings)
display(checks.round(3))
display(checks.groupby('model')[['absolute_error','covered','width','log_score']].mean().round(3))
# Agent challenge: compare kernels and mean functions using this same protocol.
# Keep the final unseen period untouched; record how many configurations you tried.


## 4. Build your portfolio
The helper below fits each stock separately, then couples endpoint forecast draws using a **shrunk historical return correlation**. This is an approximation: return correlation need not equal forecast-error correlation, and it may change. It is not a multi-output GP.

Compare correlated and independent draws. Do not treat three uncertain forecasts as automatic diversification.

Starter allocation: equal amounts in the stocks and cash. You may choose another rule, such as maximizing expected value subject to a stated loss-probability constraint. Do not choose solely by one lucky Monte Carlo draw.


In [ ]:
forecasts = [forecast(history, asset, **settings) for asset in ASSETS]
weights = np.array([.25, .25, .25, .25])  # EDIT: A, B, C, cash; nonnegative and sum to 1
values, correlation = portfolio_scenarios(history, forecasts, weights)
independent_values, _ = portfolio_scenarios(history, forecasts, weights, correlated=False)
display(pd.DataFrame(correlation, index=ASSETS, columns=ASSETS).round(2))
print('Predicted mean final value:', round(values.mean()))
print('Predicted 90% interval:', np.quantile(values, [.05, .95]).round())
print('Predicted probability of loss:', round(100*np.mean(values < 10000-1e-8), 1), '%')
plt.hist(values, bins=60, alpha=.6, density=True, label='Correlated approximation')
plt.hist(independent_values, bins=60, alpha=.4, density=True, label='Independent assumption')
plt.axvline(10000, color='black', linestyle='--'); plt.xlabel('Final portfolio value'); plt.legend(); plt.show()


## 5. Submit and lock your decision
Open the QR-code link on the projector. One teammate submits:
- Team name and percentages in A, B, C, and cash.
- Predicted probability of losing money (percent).
- One sentence explaining your allocation and its weakest assumption.

You can revise from the same browser until the instructor locks submissions. The app computes buy-and-hold values using the actual investment-date prices, not the initial price of 100.

**Stop here for the live reveal.** Did the best outcome correspond to the strongest evidence? What would repeated possible futures tell us?

Write down a reason you might reject your own model even if your portfolio makes money.


## Optional agent extension: put a prior on length scale
Change the prior probabilities over short, medium, and long length scales. Posterior model weights combine prior probabilities with each model's marginal likelihood. Other parameters and the mean choice remain fixed.

The mixture is generally not Gaussian. Sample a component first, then sample its forecast. This extension is shown for one stock; the core portfolio helper still uses the single chosen configuration above.


In [ ]:
components, probabilities = mixture_forecast(history, ASSET, lengths=(8.,25.,80.),
    prior=(.2,.6,.2), kernel=KERNEL, amplitude=AMPLITUDE, noise=NOISE, mean=MEAN)
display(pd.DataFrame({'length':[8,25,80], 'prior':[.2,.6,.2], 'posterior':probabilities}))
rng = np.random.default_rng(19)
which = rng.choice(3, 10000, p=probabilities)
terminal = np.array([np.exp(rng.normal(components[j].mean[-1], np.sqrt(components[j].covariance[-1,-1]))) for j in which])
plt.hist(terminal, bins=60, density=True); plt.xlabel('Mixture forecast: terminal price'); plt.show()


## Debrief
- Which assumptions mattered most to your allocation?
- Were your intervals for future prices, or only the latent function?
- How much did the dependence assumption affect risk?
- How would you test this on more periods without tuning on the final test set?
- Did the simulated-futures leaderboard change your interpretation of the profit winner?

References: [GPML regression chapter](https://gaussianprocess.org/gpml/chapters/RW2.pdf), [scikit-learn GPR documentation](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html).
